In [13]:
# Basic Package Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error, r2_score

# imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Non-basic package imports
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import requests

# Packages I don't understand
from fcd_torch import FCD
import rdkit
from collections import Counter
import gc
import pickle

# Add the Python_files directory to the Python path
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'Python_files'))

# Now you can import your modules
import functions_enc as f
import function_depot as fd

# Morgan Fingerpring Data Processing

In [14]:
# We are switching to the June 25 dataset
df4fp = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/MIT_LL_data4_fingerprints.csv")
print(df4fp.shape)
df4fp.head()

# Uniformity of ionization model labels
print(df4fp["Ionization_Mode"].unique())
df4fp["Ionization_Mode"] = df4fp["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df4fp["Ionization_Mode"].unique())
# Remove the N/A values in Ionization_Mode
df4fp = df4fp[df4fp["Ionization_Mode"] != "'N/A'"]
print(df4fp["Ionization_Mode"].unique())

# Remove single quotes from all string columns in df4
df4fp = df4fp.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)

# print(df4fp['Group'].nunique()) # 12
# # 12 groups
# print(df4fp['Group'].unique())
# # Now we want counts of each group
# print(df4fp['Group'].value_counts())

# This will give us the subsets with all of the relevant information
df4fp_QQpos = df4fp[df4fp['Group'] == 'Q-Orbitrap-positive'] # 2065

# Other groups with their sizes listed
# df4fp_QQneg = df4fp[df4fp['Group'] == 'Q-Orbitrap-negative'] # 1400
# df4fp_QTOFpos = df4fp[df4fp['Group'] == 'Q-TOF-positive'] # 1098
# df4fp_LTQOpos = df4fp[df4fp['Group'] == 'LTQ-Orbitrap-positive'] # 615

# All of the other are just too small to be all that useful with sizes from 12 to 286 associated spectra

# df4fp_QQpos.head()


(6121, 18)
["'positive'" "'negative'" "'Positive'" "'N/A'"]
["'positive'" "'negative'" "'N/A'"]
["'positive'" "'negative'"]


In [15]:
# # SMILES count (This is the number of true ChemNet embeddings we have)
# print(df4fp['SMILES_spectra'].nunique()) # 738 unique SMILES
# print(df4fp['SMILES_spectra'].value_counts()) # 69, 64, 58, 57, 51...

# # Morgan fingerprints count
# print(df4fp['fp'].nunique()) # 658 unique Morgan fingerprints
# print(df4fp['fp'].value_counts()) 

In [16]:
# df4fp_QQpos.head()

In [23]:
# Use the function
import function_depot as fd
df4fp_QQpos_matrix = fd.expand_fingerprints_to_matrix(df4fp_QQpos)
df4fp_QQpos_matrix = df4fp_QQpos_matrix.rename(columns={'SMILES': 'SMILES_spectra'})
df4fp_QQpos_matrix.head()

Created matrix with 2065 rows and 2048 fingerprint bits
Shape: (2065, 2049)


,SMILES_spectra,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,...,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047,bit_2048
0,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# # Save the fingerprint matrix to CSV
# df4fp_QQpos_matrix.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/Morgan_fp_df4_QQpos.csv', index=False)
# print("Saved df4fp_QQpos_matrix to Morgan_fp_df4_QQpos.csv")

Saved df4fp_QQpos_matrix to Morgan_fp_df4_QQpos.csv


In [25]:
data = pd.read_pickle('/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/bin0_1_thresh_zero_df_spectra.pkl')
data.head()

,SMILES_spectra,0.05,0.15000000000000002,0.25,0.35,0.44999999999999996,0.5499999999999999,0.6499999999999999,0.7499999999999999,0.8499999999999999,...,678.8500000000859,678.9500000000859,679.0500000000859,679.1500000000859,679.250000000086,679.350000000086,679.45,index_id,Response,log_response
0,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,273.642508,5.611823
1,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,273.642508,5.611823
2,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,273.642508,5.611823
3,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,273.642508,5.611823
4,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,273.642508,5.611823


In [27]:
# Training and validation dataset split for Morgan fingerprint encoder
data = pd.read_pickle('/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/bin0_1_thresh_zero_df_spectra.pkl')
df4fp_QQpos_matrix = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/Morgan_fp_df4_QQpos.csv')

In [28]:
data.head()

,SMILES_spectra,0.05,0.15000000000000002,0.25,0.35,0.44999999999999996,0.5499999999999999,0.6499999999999999,0.7499999999999999,0.8499999999999999,...,678.8500000000859,678.9500000000859,679.0500000000859,679.1500000000859,679.250000000086,679.350000000086,679.45,index_id,Response,log_response
0,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,273.642508,5.611823
1,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,273.642508,5.611823
2,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,273.642508,5.611823
3,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,273.642508,5.611823
4,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,273.642508,5.611823


In [29]:
df4fp_QQpos_matrix.head()

,SMILES_spectra,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,...,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047,bit_2048
0,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Train an encoder with Morgan Fingerprints

In [ ]:
# Training and validation dataset split for Morgan fingerprint encoder
data = pd.read_pickle('/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/bin0_1_thresh_zero_df_spectra.pkl')
df4fp_QQpos_matrix = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/Morgan_fp_df4_QQpos.csv')

# Merge the datasets on SMILES to get paired spectra-fingerprint data
merged_data = data.merge(df4fp_QQpos_matrix, on='SMILES_spectra', how='inner')
print(f"Merged dataset shape: {merged_data.shape}")

# Generalize the syntax
dataset = merged_data

# Count occurrences of each SMILES and keep only SMILES with at least 4 entries
counts = dataset['SMILES_spectra'].value_counts()
valid_smiles = counts[counts >= 4].index
filtered_dataset = dataset[dataset['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

np.random.seed(42)  # For reproducibility

for smiles, group in filtered_dataset.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_dataset.loc[train_indices].reset_index(drop=True)
test_data = filtered_dataset.loc[test_indices].reset_index(drop=True)

# Add an 'index' column 
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)

# Make a copy
train_data_morgan_copy = train_data.copy()
test_data_morgan_copy = test_data.copy()

# Save under correct names
train_data_morgan = train_data
test_data_morgan = test_data
# Load val_data
val_data = test_data

# Ensure we match the data correctly:
train_data = train_data_morgan
test_data = test_data_morgan
val_data = test_data
train_data_morgan_copy = train_data.copy()
test_data_morgan_copy = test_data.copy()

# Morgan fingerprint encoder training
device = fd.set_up_gpu()

# Get spectra columns and fingerprint columns
spectra_cols = [col for col in data.columns if col != 'SMILES_spectra']
fp_cols = [col for col in df4fp_QQpos_matrix.columns if col.startswith('bit_')]

# Find the start and stop indices for spectra columns in the merged data
start_idx = 1  # Skip SMILES column
stop_idx = len(spectra_cols) + 1  # Include all spectra columns

# Training set
train_embeddings_tensor, train_spectra_tensor, train_indices_tensor = fd.create_dataset_tensors(
    train_data, df4fp_QQpos_matrix, device, start_idx=start_idx, stop_idx=stop_idx)
del train_data

# Validation set
val_embeddings_tensor, val_spectra_tensor, val_indices_tensor = fd.create_dataset_tensors(
    val_data, df4fp_QQpos_matrix, device, start_idx=start_idx, stop_idx=stop_idx)
del val_data

train_data = TensorDataset(train_spectra_tensor, train_embeddings_tensor, train_indices_tensor)
val_data = TensorDataset(val_spectra_tensor, val_embeddings_tensor, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)

# Set up model parameters
input_size = len(spectra_cols)  # Number of spectra features
output_size = 2048
num_layers = 3
learning_rate = 0.0001
epochs = 100

# Create Morgan fingerprint encoder
morgan_encoder = fd.Morgan_fp_Encoder(input_size=input_size, output_size=output_size, num_layers=num_layers).to(device)

# Use BCEWithLogitsLoss since fingerprints are binary
criterion = nn.BCEWithLogitsLoss()

# Train the model
model_morgan = fd.train_model_morgan_fp_encoder(
    model=morgan_encoder,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=learning_rate,
    criterion=criterion,
    device=device
)

x_val_morgan = val_spectra_tensor
x_train_morgan = train_spectra_tensor

print("Training complete! Model can now predict Morgan fingerprints from spectra.")

Merged dataset shape: (15679, 10858)
(7824, 10859)
(7770, 10859)
Using CPU
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/dlipsey/MITLincolnLabs/.venv/lib/python3.8/site-packages/IPython/core/interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2032242/2773015861.py", line 61, in <module>
    device = fd.set_up_gpu()
  File "/home/dlipsey/MITLincolnLabs/Python_files/function_depot.py", line 381, in set_up_gpu
  File "/home/dlipsey/MITLincolnLabs/.venv/lib/python3.8/site-packages/torch/cuda/__init__.py", line 878, in current_device
    _lazy_init()
  File "/home/dlipsey/MITLincolnLabs/.venv/lib/python3.8/site-packages/torch/cuda/__init__.py", line 314, in _lazy_init
    torch._C._cuda_init()
RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/dlipsey/MITLincolnLab